In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df= pd.read_csv("imdb_dataset.csv")
df.shape

In [ ]:
df.isnull().sum()
df.drop_duplicates(inplace=True)

In [ ]:
# data preprocessing 
# 1- covert into lowercase 
 df["review"]=df["review"].str.lower()



In [ ]:
# remove urls
import re

def remove_urls(text):
text=re.sub(r"http\S+","",text) # pattern replacemnt, text
return text

df['review']=df['review'].apply(remove_urls)


In [ ]:
# rmoving punctuations

def remove_punc(text):
text=re.sub(r"[^A-Za-z0-9\s]","",text)
return text

df['review']=df['review'].apply(remove_pucn)



In [ ]:
# remove html

def remove_html(text):
text=re.sub(r"<.*?>","",text)
return text

df['review']=df['review'].apply(remove_html)



In [ ]:
# removing stop words
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords)



In [ ]:
from nltk.tokenize import word_tokenize
from nltk.coprus import stop_words

def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stop_words.words("english)
     for word in tokens:
         if word in stop_words:
           text =text.replece(word,"")
    return text
   
df['review']=df['review'].apply(remove_stopwords)



In [ ]:
df.head()

In [ ]:
## 6-stemming
#running -> run 

from nltk.stem import PorterStemmer

def stemming (text):
    ps=PorterStemmer()
    stemmed_words=[]
    token=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    return"".join (stemmed_words)

df['review']=df['review'].apply(stemming)

In [ ]:
df.head()

In [ ]:
# encoding
from sklearn.preprocessing import LabelEncoder
le= LabelEncoder()
df['sentiment'] = le.LabelEncoder(df['sentiment'])
y=df['sentiment'] 



In [ ]:
# Vectorization 
from sklearn.feature_exteraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)

X=tf.fit_transform(df['review'])




In [ ]:
# prepare the dataset 
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y.test_size=0.2,random_state=42)


In [ ]:
# into numpy array
X_train= X_train.toarray()
X_test=X_test.toarray()


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float(),
)
 
test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float(),
)
 

In [ ]:
# loaders
train_loader=DataLoader(train_set,batch_size=64,shuffle=True)
test_loader=DataLoader(test_set,batch_size=64,shuffle=True)

In [ ]:
# Build RNN
import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
     def __init__(self,input_size,hidden_size=128, num_layers=1):
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layers=num_layers

        self.rnn = nn.RNN(input_size,hidden_size,num_layers, batch_first=True)
        # fully connected num_layers
        self.fc=nn.Linear(hidden_size,1) # many to one architecure 

    def forward(self,x):
        h0= torch.zeros(sef.num_layers,x.size(0),self.hidden_size)

        out,_= self.rnn(x,h0)

         # 1- return hidden state of all timestemps dim-> batch,sequence_lenth
         # 2- final hidden state of the last timestemps
        final_output=self.fc(out[:,-1:]) # simliar to probability 
        return final_output








In [ ]:
input_size=X_train.shape[1]

model=RNN(input_size)

criterian=nn.BCELoss()
optimizer=optim.Adma(model.parameters())


In [ ]:
# Training 
# unsqueeze:add one dimension  squeeze: reduce one dimension
epochs=10
for epoch in range(epochs):
    model.train()

    for Xb,yb in train_loader:
        optimizer.zero_grad()
        Xb=Xb.unsqueeze(1) # add signleton direction 
        outputs=model(xb) # output 2d(batch_size,1)
        final_outputs=torch.sigmoid(outputs.squeeze())
        loss=criterian(final_outputs,yb)# compute loss
        loss.backward() # BP
        optimizer.step()# adjusting parameters

    print(f"Epoch : {epoch+1}/{epochs} and the loss value :{loss.item()}")


In [ ]:
# Evaluation Model
model.eval()
with torch.no_grad():
 correct_value=0,
 total_value=0
  for Xb,yb in test_loader:
    Xb=Xb.unsqueeze(1)

    outputs=model(Xb)
    predicted_value=(torch.sigmoid(outputs.squeeze())>0.5).float()

    total_value=yb.size(0)
    correct_value=(predicted_value==yb).sum().item()

print(f" Accuracy -> {correct_value/total_value  * 100 }")

